In [ ]:
from __future__ import annotations

from itertools import product
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from catboost import CatBoostRegressor, Pool

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)
sns.set_theme(style="whitegrid")




In [ ]:
# -----------------------------------------------------------------------------
# Config
# -----------------------------------------------------------------------------

OUTPUT_DIR = Path.cwd() / "notebook_exports"
ROUND_DATA_PATH = OUTPUT_DIR / "round_level_modeling_dataset_v1.parquet"

TEST_SIZE = 0.20
VALID_SIZE_WITHIN_TRAIN = 0.20
RANDOM_STATE = 42

USE_LAYOUT_ID = True
USE_DIVISION = True
REQUIRE_WEATHER_AVAILABLE = True

MPH_TO_MPS = 0.44704
MPS_TO_MPH = 2.23694

TARGET_COL = "actual_round_strokes"

CATBOOST_PARAMS = {
    "loss_function": "RMSE",
    "eval_metric": "RMSE",
    "iterations": 12000,
    "depth": 4,
    "learning_rate": 0.03,
    "l2_leaf_reg": 15.0,
    "random_seed": RANDOM_STATE,
    "verbose": 250,
    "bootstrap_type": "Bernoulli",
    "subsample": 0.7,
    "max_ctr_complexity": 1,
    "store_all_simple_ctr": False,
    "task_type": "CPU",
}

EARLY_STOPPING_ROUNDS = 500
IDEAL_PRECIP_REFERENCE_MM = 0.0

print("Using local round dataset:", ROUND_DATA_PATH)

In [ ]:
# -----------------------------------------------------------------------------
# Load data
# -----------------------------------------------------------------------------

round_level_df = pd.read_parquet(ROUND_DATA_PATH).copy()

print("Round-level shape:", round_level_df.shape)
round_level_df.head()


In [ ]:
# -----------------------------------------------------------------------------
# Helpers
# -----------------------------------------------------------------------------

def wind_speed_bucket_from_mps(speed_mps: float | None) -> str:
    if speed_mps is None or pd.isna(speed_mps):
        return "unknown"
    if speed_mps < 2.0:
        return "calm"
    if speed_mps < 5.0:
        return "light"
    if speed_mps < 8.0:
        return "moderate"
    if speed_mps < 12.0:
        return "strong"
    return "very_strong"


def wind_gust_bucket_from_mps(speed_mps: float | None) -> str:
    if speed_mps is None or pd.isna(speed_mps):
        return "unknown"
    if speed_mps < 3.0:
        return "low"
    if speed_mps < 6.0:
        return "mild"
    if speed_mps < 10.0:
        return "high"
    return "very_high"


def wind_speed_bucket_from_mph(speed_mph: float | None) -> str:
    if speed_mph is None or pd.isna(speed_mph):
        return "unknown"
    return wind_speed_bucket_from_mps(speed_mph * MPH_TO_MPS)


def wind_gust_bucket_from_mph(speed_mph: float | None) -> str:
    if speed_mph is None or pd.isna(speed_mph):
        return "unknown"
    return wind_gust_bucket_from_mps(speed_mph * MPH_TO_MPS)


def regression_metrics(y_true: pd.Series, y_pred: np.ndarray) -> dict[str, float]:
    return {
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "rmse": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "r2": float(r2_score(y_true, y_pred)),
    }


def build_pool_from_prepared_df(
    df: pd.DataFrame,
    *,
    feature_cols: list[str],
    categorical_features: list[str],
    target_col: str | None = None,
) -> Pool:
    cat_idx = [feature_cols.index(c) for c in categorical_features]

    if target_col is not None:
        return Pool(
            data=df[feature_cols],
            label=df[target_col],
            cat_features=cat_idx,
        )

    return Pool(
        data=df[feature_cols],
        cat_features=cat_idx,
    )


def _prep_single_frame(
    df: pd.DataFrame,
    *,
    numeric_features: list[str],
    categorical_features: list[str],
    target_col: str,
) -> pd.DataFrame:
    out = df.copy()

    required = [target_col] + numeric_features + categorical_features
    missing = [c for c in required if c not in out.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    for col in numeric_features + [target_col]:
        out[col] = pd.to_numeric(out[col], errors="coerce")

    out = out.dropna(subset=numeric_features + [target_col]).copy()

    for col in categorical_features:
        out[col] = out[col].astype("string").fillna("__MISSING__").astype(str)

    return out


def prepare_round_model_data(
    *,
    train_df: pd.DataFrame,
    valid_df: pd.DataFrame,
    test_df: pd.DataFrame,
    numeric_features: list[str],
    categorical_features: list[str],
    target_col: str,
):
    train_prepped = _prep_single_frame(
        train_df,
        numeric_features=numeric_features,
        categorical_features=categorical_features,
        target_col=target_col,
    )
    valid_prepped = _prep_single_frame(
        valid_df,
        numeric_features=numeric_features,
        categorical_features=categorical_features,
        target_col=target_col,
    )
    test_prepped = _prep_single_frame(
        test_df,
        numeric_features=numeric_features,
        categorical_features=categorical_features,
        target_col=target_col,
    )

    feature_cols = list(numeric_features) + list(categorical_features)

    train_pool = build_pool_from_prepared_df(
        train_prepped,
        feature_cols=feature_cols,
        categorical_features=categorical_features,
        target_col=target_col,
    )
    valid_pool = build_pool_from_prepared_df(
        valid_prepped,
        feature_cols=feature_cols,
        categorical_features=categorical_features,
        target_col=target_col,
    )
    test_pool = build_pool_from_prepared_df(
        test_prepped,
        feature_cols=feature_cols,
        categorical_features=categorical_features,
        target_col=target_col,
    )

    return (
        train_prepped,
        valid_prepped,
        test_prepped,
        train_pool,
        valid_pool,
        test_pool,
        feature_cols,
    )


def effect_summary(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for metric in [
        "estimated_wind_effect_vs_ideal",
        "estimated_temperature_effect_vs_ideal",
        "estimated_precip_effect_vs_ideal",
        "estimated_total_weather_effect_vs_ideal",
    ]:
        rows.append(
            {
                "metric": metric,
                "mean": float(df[metric].mean()),
                "median": float(df[metric].median()),
                "std": float(df[metric].std()),
                "p10": float(df[metric].quantile(0.10)),
                "p90": float(df[metric].quantile(0.90)),
            }
        )
    return pd.DataFrame(rows)


def build_weather_scenario_df(
    base_df: pd.DataFrame,
    *,
    wind_speed_mph: float,
    wind_gust_mph: float,
    temperature_c: float,
    precip_mm: float,
) -> pd.DataFrame:
    out = base_df.copy()

    wind_speed_mps = wind_speed_mph * MPH_TO_MPS
    wind_gust_mps = wind_gust_mph * MPH_TO_MPS

    out["round_wind_speed_mps_mean"] = wind_speed_mps
    out["round_wind_speed_mps_max"] = wind_speed_mps
    out["round_wind_gust_mps_mean"] = wind_gust_mps
    out["round_wind_gust_mps_max"] = wind_gust_mps

    out["round_wind_speed_bucket"] = wind_speed_bucket_from_mph(wind_speed_mph)
    out["round_wind_gust_bucket"] = wind_gust_bucket_from_mph(wind_gust_mph)

    out["round_temp_c_mean"] = float(temperature_c)
    out["round_precip_mm_sum"] = float(precip_mm)
    out["round_precip_mm_mean"] = float(precip_mm)

    return out


def build_partial_dependence_curve(
    *,
    base_df: pd.DataFrame,
    model,
    feature_cols: list[str],
    categorical_features: list[str],
    vary_col: str,
    grid_values: list[float],
    update_fn=None,
) -> pd.DataFrame:
    rows = []

    for grid_value in grid_values:
        cf_df = base_df.copy()
        cf_df[vary_col] = grid_value

        if update_fn is not None:
            cf_df = update_fn(cf_df, grid_value)

        cf_pool = build_pool_from_prepared_df(
            cf_df,
            feature_cols=feature_cols,
            categorical_features=categorical_features,
            target_col=None,
        )
        preds = model.predict(cf_pool)

        rows.append(
            {
                "grid_value": grid_value,
                "prediction_mean": float(np.mean(preds)),
                "prediction_median": float(np.median(preds)),
                "prediction_std": float(np.std(preds)),
                "prediction_p10": float(np.quantile(preds, 0.10)),
                "prediction_p90": float(np.quantile(preds, 0.90)),
            }
        )

    return pd.DataFrame(rows)


def build_effect_pdp_curve(
    *,
    base_df: pd.DataFrame,
    model,
    feature_cols: list[str],
    categorical_features: list[str],
    update_actual_fn,
    update_reference_fn,
    grid_values: list[float],
) -> pd.DataFrame:
    rows = []

    for grid_value in grid_values:
        actual_df = update_actual_fn(base_df.copy(), grid_value)
        reference_df = update_reference_fn(base_df.copy(), grid_value)

        actual_pool = build_pool_from_prepared_df(
            actual_df,
            feature_cols=feature_cols,
            categorical_features=categorical_features,
            target_col=None,
        )
        reference_pool = build_pool_from_prepared_df(
            reference_df,
            feature_cols=feature_cols,
            categorical_features=categorical_features,
            target_col=None,
        )

        actual_pred = model.predict(actual_pool)
        reference_pred = model.predict(reference_pool)

        effect_values = actual_pred - reference_pred

        rows.append(
            {
                "grid_value": grid_value,
                "effect_mean": float(np.mean(effect_values)),
                "effect_median": float(np.median(effect_values)),
                "effect_std": float(np.std(effect_values)),
                "effect_p10": float(np.quantile(effect_values, 0.10)),
                "effect_p90": float(np.quantile(effect_values, 0.90)),
            }
        )

    return pd.DataFrame(rows)


In [ ]:
# -----------------------------------------------------------------------------
# Feature setup
# -----------------------------------------------------------------------------

model_df = round_level_df.copy()

if REQUIRE_WEATHER_AVAILABLE and "weather_available_flag" in model_df.columns:
    model_df = model_df[model_df["weather_available_flag"] == True].copy()

if "round_wind_speed_bucket" not in model_df.columns:
    model_df["round_wind_speed_bucket"] = model_df["round_wind_speed_mps_mean"].map(wind_speed_bucket_from_mps)

if "round_wind_gust_bucket" not in model_df.columns:
    model_df["round_wind_gust_bucket"] = model_df["round_wind_gust_mps_mean"].map(wind_gust_bucket_from_mps)

base_numeric_features = [
    "player_rating",
    "round_number",
    "hole_count",
    "round_total_hole_length",
    "round_avg_hole_length",
    "round_total_par",
    "round_avg_hole_par",
    "round_length_over_par",
    "round_wind_speed_mps_mean",
    "round_temp_c_mean",
    "round_precip_mm_sum",
]

base_categorical_features = [
    "course_id",
    #"round_wind_speed_bucket",
    #"round_wind_gust_bucket",
]

if USE_DIVISION:
    base_categorical_features.append("division")

if USE_LAYOUT_ID:
    base_categorical_features.append("layout_id")

interaction_like_cols = [
    "wind_x_round_length",
    "gust_x_round_length",
    "wind_x_player_rating",
]
present_interaction_cols = [c for c in interaction_like_cols if c in model_df.columns]

print("Present hand-built interaction columns being ignored:", present_interaction_cols)

MODEL_EXPERIMENT = {
    "name": "no_weather_interactions_empirical_ideal_baseline_core_weather_only",
    "numeric_features": base_numeric_features,
    "categorical_features": base_categorical_features,
}

MODEL_EXPERIMENT


In [ ]:
# -----------------------------------------------------------------------------
# Train / validation / test split
# -----------------------------------------------------------------------------

train_full_df, test_df = train_test_split(
    model_df,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
)

train_df, valid_df = train_test_split(
    train_full_df,
    test_size=VALID_SIZE_WITHIN_TRAIN,
    random_state=RANDOM_STATE,
)

print("Train rows:", len(train_df))
print("Valid rows:", len(valid_df))
print("Test rows:", len(test_df))


In [ ]:
# -----------------------------------------------------------------------------
# Train single one-stage weather model
# -----------------------------------------------------------------------------

experiment = MODEL_EXPERIMENT
numeric_features = experiment["numeric_features"]
categorical_features = experiment["categorical_features"]

(
    train_prepped,
    valid_prepped,
    test_prepped,
    train_pool,
    valid_pool,
    test_pool,
    feature_cols,
) = prepare_round_model_data(
    train_df=train_df,
    valid_df=valid_df,
    test_df=test_df,
    numeric_features=numeric_features,
    categorical_features=categorical_features,
    target_col=TARGET_COL,
)

model = CatBoostRegressor(**CATBOOST_PARAMS)
model.fit(
    train_pool,
    eval_set=valid_pool,
    use_best_model=True,
    early_stopping_rounds=EARLY_STOPPING_ROUNDS,
)

train_pred = model.predict(train_pool)
valid_pred = model.predict(valid_pool)
test_pred = model.predict(test_pool)

train_metrics = regression_metrics(train_prepped[TARGET_COL], train_pred)
valid_metrics = regression_metrics(valid_prepped[TARGET_COL], valid_pred)
test_metrics = regression_metrics(test_prepped[TARGET_COL], test_pred)

best_iteration = getattr(model, "best_iteration_", None)

model_metrics = pd.DataFrame(
    [
        {"split": "train", **train_metrics},
        {"split": "valid", **valid_metrics},
        {"split": "test", **test_metrics},
    ]
)
model_metrics["experiment"] = experiment["name"]
model_metrics["best_iteration"] = best_iteration
model_metrics["feature_count"] = len(feature_cols)

model_metrics


In [ ]:
# -----------------------------------------------------------------------------
# Feature importance
# -----------------------------------------------------------------------------

feature_importance_df = model.get_feature_importance(
    train_pool,
    prettified=True,
).rename(columns={"Feature Id": "feature", "Importances": "importance"})

feature_importance_df.head(25)


In [ ]:
# -----------------------------------------------------------------------------
# Empirical ideal-weather search config
# -----------------------------------------------------------------------------

ideal_search_config = {
    "wind_speed_mph": [0.0, 1.0, 2.0, 3.0, 5.0, 8.0],
    "wind_gust_mph": [0.0, 2.0, 3.0, 5.0, 8.0, 12.0],
    "temperature_c": [8.0, 11.0, 14.0, 17.0, 18.0, 19.0, 22.0, 25.0, 28.0, 31.0, 34.0, 37.0],
}

ideal_search_config


In [ ]:
# -----------------------------------------------------------------------------
# Build candidate search grid
# -----------------------------------------------------------------------------

search_rows = []
for wind_speed_mph, wind_gust_mph, temperature_c in product(
    ideal_search_config["wind_speed_mph"],
    ideal_search_config["wind_gust_mph"],
    ideal_search_config["temperature_c"],
):
    if wind_gust_mph < wind_speed_mph:
        continue

    search_rows.append(
        {
            "wind_speed_mph": wind_speed_mph,
            "wind_gust_mph": wind_gust_mph,
            "temperature_c": temperature_c,
            "temperature_f": (temperature_c * 9.0 / 5.0) + 32.0,
            "precip_mm": IDEAL_PRECIP_REFERENCE_MM,
        }
    )

ideal_weather_grid_df = pd.DataFrame(search_rows)
print("Candidate weather combinations:", len(ideal_weather_grid_df))
ideal_weather_grid_df.head(20)


In [ ]:
# -----------------------------------------------------------------------------
# Choose a base sample for the ideal-weather search
# -----------------------------------------------------------------------------

ideal_search_base_df = test_prepped.copy()

MAX_BASE_ROWS = 25000
if len(ideal_search_base_df) > MAX_BASE_ROWS:
    ideal_search_base_df = ideal_search_base_df.sample(MAX_BASE_ROWS, random_state=RANDOM_STATE).copy()

print("Rows used for ideal-weather search:", len(ideal_search_base_df))


In [ ]:
# -----------------------------------------------------------------------------
# Run empirical ideal-weather search
# -----------------------------------------------------------------------------

ideal_weather_results = []

for _, candidate in ideal_weather_grid_df.iterrows():
    scenario_df = build_weather_scenario_df(
        ideal_search_base_df,
        wind_speed_mph=float(candidate["wind_speed_mph"]),
        wind_gust_mph=float(candidate["wind_gust_mph"]),
        temperature_c=float(candidate["temperature_c"]),
        precip_mm=IDEAL_PRECIP_REFERENCE_MM,
    )

    scenario_pool = build_pool_from_prepared_df(
        scenario_df,
        feature_cols=feature_cols,
        categorical_features=categorical_features,
        target_col=None,
    )

    scenario_pred = model.predict(scenario_pool)

    ideal_weather_results.append(
        {
            "wind_speed_mph": float(candidate["wind_speed_mph"]),
            "wind_gust_mph": float(candidate["wind_gust_mph"]),
            "temperature_c": float(candidate["temperature_c"]),
            "temperature_f": float(candidate["temperature_f"]),
            "precip_mm": IDEAL_PRECIP_REFERENCE_MM,
            "predicted_round_strokes_mean": float(np.mean(scenario_pred)),
            "predicted_round_strokes_median": float(np.median(scenario_pred)),
            "predicted_round_strokes_std": float(np.std(scenario_pred)),
        }
    )

ideal_weather_results_df = (
    pd.DataFrame(ideal_weather_results)
    .sort_values(
        ["predicted_round_strokes_mean", "predicted_round_strokes_median"],
        ascending=[True, True],
    )
    .reset_index(drop=True)
)

ideal_weather_results_df.head(20)



In [ ]:
# -----------------------------------------------------------------------------
# Best empirical ideal-weather baseline
# -----------------------------------------------------------------------------

empirical_ideal_weather_baseline = ideal_weather_results_df.iloc[0].to_dict()
empirical_ideal_weather_baseline


In [ ]:
# -----------------------------------------------------------------------------
# Top candidate visualization
# -----------------------------------------------------------------------------

plt.figure(figsize=(10, 6))
plot_df = ideal_weather_results_df.head(25).copy()

sns.scatterplot(
    data=plot_df,
    x="temperature_f",
    y="predicted_round_strokes_mean",
    hue="wind_speed_mph",
    size="wind_gust_mph",
    sizes=(40, 220),
)
plt.title("Top Empirical Ideal-Weather Candidates")
plt.xlabel("Temperature (F)")
plt.ylabel("Mean predicted round strokes")
plt.tight_layout()
plt.show()


In [ ]:
# -----------------------------------------------------------------------------
# Freeze ideal baseline values for downstream scoring
# -----------------------------------------------------------------------------

IDEAL_WIND_REFERENCE_MPH = float(empirical_ideal_weather_baseline["wind_speed_mph"])
IDEAL_WIND_GUST_REFERENCE_MPH = float(empirical_ideal_weather_baseline["wind_gust_mph"])
IDEAL_TEMPERATURE_REFERENCE_C = float(empirical_ideal_weather_baseline["temperature_c"])

ideal_reference_conditions_df = pd.DataFrame(
    [
        {
            "wind_speed_reference_mph": IDEAL_WIND_REFERENCE_MPH,
            "wind_gust_reference_mph": IDEAL_WIND_GUST_REFERENCE_MPH,
            "temperature_reference_c": IDEAL_TEMPERATURE_REFERENCE_C,
            "temperature_reference_f": (IDEAL_TEMPERATURE_REFERENCE_C * 9.0 / 5.0) + 32.0,
            "precip_reference_mm": IDEAL_PRECIP_REFERENCE_MM,
        }
    ]
)

ideal_reference_conditions_df


In [ ]:
# -----------------------------------------------------------------------------
# Score observed rows
# -----------------------------------------------------------------------------

scored_df = test_prepped.copy()
scored_df["predicted_round_strokes"] = test_pred
scored_df["round_residual_strokes"] = (
    scored_df["actual_round_strokes"] - scored_df["predicted_round_strokes"]
)


In [ ]:
# -----------------------------------------------------------------------------
# Wind effect vs ideal:
# keep observed temp/precip, replace wind bundle with ideal wind bundle
# -----------------------------------------------------------------------------

wind_reference_df = test_prepped.copy()
wind_reference_df["round_wind_speed_mps_mean"] = IDEAL_WIND_REFERENCE_MPH * MPH_TO_MPS
wind_reference_df["round_wind_speed_mps_max"] = IDEAL_WIND_REFERENCE_MPH * MPH_TO_MPS
wind_reference_df["round_wind_gust_mps_mean"] = IDEAL_WIND_GUST_REFERENCE_MPH * MPH_TO_MPS
wind_reference_df["round_wind_gust_mps_max"] = IDEAL_WIND_GUST_REFERENCE_MPH * MPH_TO_MPS
wind_reference_df["round_wind_speed_bucket"] = wind_speed_bucket_from_mph(IDEAL_WIND_REFERENCE_MPH)
wind_reference_df["round_wind_gust_bucket"] = wind_gust_bucket_from_mph(IDEAL_WIND_GUST_REFERENCE_MPH)

wind_reference_pool = build_pool_from_prepared_df(
    wind_reference_df,
    feature_cols=feature_cols,
    categorical_features=categorical_features,
    target_col=None,
)
wind_reference_pred = model.predict(wind_reference_pool)

scored_df["predicted_round_strokes_wind_reference_ideal"] = wind_reference_pred
scored_df["estimated_wind_effect_vs_ideal"] = (
    scored_df["predicted_round_strokes"] - scored_df["predicted_round_strokes_wind_reference_ideal"]
)


In [ ]:
# -----------------------------------------------------------------------------
# Temperature effect vs ideal:
# keep observed wind/precip, replace temp with ideal temp
# -----------------------------------------------------------------------------

temperature_reference_df = test_prepped.copy()
temperature_reference_df["round_temp_c_mean"] = IDEAL_TEMPERATURE_REFERENCE_C

temperature_reference_pool = build_pool_from_prepared_df(
    temperature_reference_df,
    feature_cols=feature_cols,
    categorical_features=categorical_features,
    target_col=None,
)
temperature_reference_pred = model.predict(temperature_reference_pool)

scored_df["predicted_round_strokes_temperature_reference_ideal"] = temperature_reference_pred
scored_df["estimated_temperature_effect_vs_ideal"] = (
    scored_df["predicted_round_strokes"] - scored_df["predicted_round_strokes_temperature_reference_ideal"]
)


In [ ]:
# -----------------------------------------------------------------------------
# Precipitation effect vs ideal:
# keep observed wind/temp, replace precip with ideal precip (= 0)
# -----------------------------------------------------------------------------

precip_reference_df = test_prepped.copy()
precip_reference_df["round_precip_mm_sum"] = IDEAL_PRECIP_REFERENCE_MM
precip_reference_df["round_precip_mm_mean"] = IDEAL_PRECIP_REFERENCE_MM

precip_reference_pool = build_pool_from_prepared_df(
    precip_reference_df,
    feature_cols=feature_cols,
    categorical_features=categorical_features,
    target_col=None,
)
precip_reference_pred = model.predict(precip_reference_pool)

scored_df["predicted_round_strokes_precip_reference_ideal"] = precip_reference_pred
scored_df["estimated_precip_effect_vs_ideal"] = (
    scored_df["predicted_round_strokes"] - scored_df["predicted_round_strokes_precip_reference_ideal"]
)


In [ ]:
# -----------------------------------------------------------------------------
# Total weather effect vs ideal:
# replace wind/temp/precip with ideal baseline
# -----------------------------------------------------------------------------

total_weather_reference_df = build_weather_scenario_df(
    test_prepped,
    wind_speed_mph=IDEAL_WIND_REFERENCE_MPH,
    wind_gust_mph=IDEAL_WIND_GUST_REFERENCE_MPH,
    temperature_c=IDEAL_TEMPERATURE_REFERENCE_C,
    precip_mm=IDEAL_PRECIP_REFERENCE_MM,
)

total_weather_reference_pool = build_pool_from_prepared_df(
    total_weather_reference_df,
    feature_cols=feature_cols,
    categorical_features=categorical_features,
    target_col=None,
)
total_weather_reference_pred = model.predict(total_weather_reference_pool)

scored_df["predicted_round_strokes_total_weather_reference_ideal"] = total_weather_reference_pred
scored_df["estimated_total_weather_effect_vs_ideal"] = (
    scored_df["predicted_round_strokes"] - scored_df["predicted_round_strokes_total_weather_reference_ideal"]
)

print("Scored rows:", len(scored_df))
scored_df.head()



In [ ]:
weather_effect_summary_df = effect_summary(scored_df)
weather_effect_summary_df


In [ ]:
# -----------------------------------------------------------------------------
# Grouped summaries
# -----------------------------------------------------------------------------

summary_df = scored_df.copy()
summary_df["round_wind_speed_mph_mean"] = summary_df["round_wind_speed_mps_mean"] * MPS_TO_MPH
summary_df["round_temp_f_mean"] = (summary_df["round_temp_c_mean"] * 9.0 / 5.0) + 32.0
summary_df["precip_flag"] = summary_df["round_precip_mm_sum"].fillna(0).gt(0).map(
    {True: "Precip", False: "No Precip"}
)

wind_bucket_summary = (
    summary_df.groupby("round_wind_speed_bucket", dropna=False)
    .agg(
        rows=("player_key", "size"),
        wind_speed_mph_mean=("round_wind_speed_mph_mean", "mean"),
        actual_round_strokes_mean=("actual_round_strokes", "mean"),
        predicted_round_strokes_mean=("predicted_round_strokes", "mean"),
        estimated_wind_effect_mean=("estimated_wind_effect_vs_ideal", "mean"),
        estimated_total_weather_effect_mean=("estimated_total_weather_effect_vs_ideal", "mean"),
    )
    .reset_index()
)

temperature_band_summary = (
    summary_df.assign(
        temperature_band_f=pd.cut(
            summary_df["round_temp_f_mean"],
            bins=[-np.inf, 41, 50, 60, 70, 80, np.inf],
            labels=["<41F", "41-49F", "50-59F", "60-69F", "70-79F", "80F+"],
        )
    )
    .groupby("temperature_band_f", dropna=False)
    .agg(
        rows=("player_key", "size"),
        temp_f_mean=("round_temp_f_mean", "mean"),
        estimated_temperature_effect_mean=("estimated_temperature_effect_vs_ideal", "mean"),
        estimated_total_weather_effect_mean=("estimated_total_weather_effect_vs_ideal", "mean"),
    )
    .reset_index()
)

precip_summary = (
    summary_df.groupby("precip_flag", dropna=False)
    .agg(
        rows=("player_key", "size"),
        precip_mm_mean=("round_precip_mm_sum", "mean"),
        estimated_precip_effect_mean=("estimated_precip_effect_vs_ideal", "mean"),
        estimated_total_weather_effect_mean=("estimated_total_weather_effect_vs_ideal", "mean"),
    )
    .reset_index()
)

print("Wind bucket summary")
display(wind_bucket_summary)

print("Temperature-band summary")
display(temperature_band_summary)

print("Precipitation summary")
display(precip_summary)


In [ ]:
stability_summary_df = pd.DataFrame(
    [
        {
            "metric": "wind_effect_std",
            "value": float(scored_df["estimated_wind_effect_vs_ideal"].std()),
        },
        {
            "metric": "temperature_effect_std",
            "value": float(scored_df["estimated_temperature_effect_vs_ideal"].std()),
        },
        {
            "metric": "precip_effect_std",
            "value": float(scored_df["estimated_precip_effect_vs_ideal"].std()),
        },
        {
            "metric": "total_weather_effect_std",
            "value": float(scored_df["estimated_total_weather_effect_vs_ideal"].std()),
        },
        {
            "metric": "wind_effect_abs_p95",
            "value": float(scored_df["estimated_wind_effect_vs_ideal"].abs().quantile(0.95)),
        },
        {
            "metric": "temperature_effect_abs_p95",
            "value": float(scored_df["estimated_temperature_effect_vs_ideal"].abs().quantile(0.95)),
        },
        {
            "metric": "precip_effect_abs_p95",
            "value": float(scored_df["estimated_precip_effect_vs_ideal"].abs().quantile(0.95)),
        },
        {
            "metric": "total_weather_effect_abs_p95",
            "value": float(scored_df["estimated_total_weather_effect_vs_ideal"].abs().quantile(0.95)),
        },
    ]
)

stability_summary_df


In [ ]:
subgroup_robustness_df = (
    scored_df.groupby("division", dropna=False)
    .agg(
        rows=("player_key", "size"),
        wind_effect_mean=("estimated_wind_effect_vs_ideal", "mean"),
        temp_effect_mean=("estimated_temperature_effect_vs_ideal", "mean"),
        precip_effect_mean=("estimated_precip_effect_vs_ideal", "mean"),
        total_weather_effect_mean=("estimated_total_weather_effect_vs_ideal", "mean"),
    )
    .reset_index()
    .sort_values("rows", ascending=False)
)

subgroup_robustness_df


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

sns.histplot(scored_df["estimated_wind_effect_vs_ideal"], bins=30, ax=axes[0, 0], color="#1f77b4")
axes[0, 0].axvline(0.0, color="black", linestyle="--", linewidth=1)
axes[0, 0].set_title("Wind Effect vs Ideal")
axes[0, 0].set_xlabel("strokes")

sns.histplot(scored_df["estimated_temperature_effect_vs_ideal"], bins=30, ax=axes[0, 1], color="#ff7f0e")
axes[0, 1].axvline(0.0, color="black", linestyle="--", linewidth=1)
axes[0, 1].set_title("Temperature Effect vs Ideal")
axes[0, 1].set_xlabel("strokes")

sns.histplot(scored_df["estimated_precip_effect_vs_ideal"], bins=30, ax=axes[1, 0], color="#2ca02c")
axes[1, 0].axvline(0.0, color="black", linestyle="--", linewidth=1)
axes[1, 0].set_title("Precipitation Effect vs Ideal")
axes[1, 0].set_xlabel("strokes")

sns.histplot(scored_df["estimated_total_weather_effect_vs_ideal"], bins=30, ax=axes[1, 1], color="#d62728")
axes[1, 1].axvline(0.0, color="black", linestyle="--", linewidth=1)
axes[1, 1].set_title("Total Weather Effect vs Ideal")
axes[1, 1].set_xlabel("strokes")

plt.tight_layout()
plt.show()


In [ ]:
pdp_base_df = scored_df.copy()
print("Rows in effect-PDP base frame:", len(pdp_base_df))


In [ ]:
# -----------------------------------------------------------------------------
# Wind effect vs ideal PDP
# -----------------------------------------------------------------------------

wind_grid_mph = [0, 2, 5, 8, 12, 16, 20, 25, 30]
wind_grid_mps = [x * MPH_TO_MPS for x in wind_grid_mph]

IDEAL_GUST_GAP_MPH = max(0.0, IDEAL_WIND_GUST_REFERENCE_MPH - IDEAL_WIND_REFERENCE_MPH)


def candidate_gust_from_wind_mph(wind_speed_mph: float) -> float:
    return max(wind_speed_mph, wind_speed_mph + IDEAL_GUST_GAP_MPH)


def update_wind_actual_df(cf_df: pd.DataFrame, grid_value_mps: float) -> pd.DataFrame:
    out = cf_df.copy()

    wind_speed_mph = grid_value_mps * MPS_TO_MPH
    wind_gust_mph = candidate_gust_from_wind_mph(wind_speed_mph)

    out["round_wind_speed_mps_mean"] = grid_value_mps
    out["round_wind_speed_mps_max"] = grid_value_mps
    out["round_wind_gust_mps_mean"] = wind_gust_mph * MPH_TO_MPS
    out["round_wind_gust_mps_max"] = wind_gust_mph * MPH_TO_MPS

    out["round_wind_speed_bucket"] = wind_speed_bucket_from_mph(wind_speed_mph)
    out["round_wind_gust_bucket"] = wind_gust_bucket_from_mph(wind_gust_mph)
    return out


def update_wind_reference_df(cf_df: pd.DataFrame, grid_value_mps: float) -> pd.DataFrame:
    out = cf_df.copy()

    out["round_wind_speed_mps_mean"] = IDEAL_WIND_REFERENCE_MPH * MPH_TO_MPS
    out["round_wind_speed_mps_max"] = IDEAL_WIND_REFERENCE_MPH * MPH_TO_MPS
    out["round_wind_gust_mps_mean"] = IDEAL_WIND_GUST_REFERENCE_MPH * MPH_TO_MPS
    out["round_wind_gust_mps_max"] = IDEAL_WIND_GUST_REFERENCE_MPH * MPH_TO_MPS

    out["round_wind_speed_bucket"] = wind_speed_bucket_from_mph(IDEAL_WIND_REFERENCE_MPH)
    out["round_wind_gust_bucket"] = wind_gust_bucket_from_mph(IDEAL_WIND_GUST_REFERENCE_MPH)
    return out


wind_effect_pdp_df = build_effect_pdp_curve(
    base_df=pdp_base_df,
    model=model,
    feature_cols=feature_cols,
    categorical_features=categorical_features,
    update_actual_fn=update_wind_actual_df,
    update_reference_fn=update_wind_reference_df,
    grid_values=wind_grid_mps,
)

wind_effect_pdp_df["wind_speed_mph"] = wind_effect_pdp_df["grid_value"] * MPS_TO_MPH
wind_effect_pdp_df


In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(
    wind_effect_pdp_df["wind_speed_mph"],
    wind_effect_pdp_df["effect_mean"],
    marker="o",
    linewidth=2.5,
    color="steelblue",
)
plt.fill_between(
    wind_effect_pdp_df["wind_speed_mph"],
    wind_effect_pdp_df["effect_p10"],
    wind_effect_pdp_df["effect_p90"],
    color="steelblue",
    alpha=0.15,
)
plt.axhline(0.0, color="black", linestyle="--", linewidth=1)
plt.axvline(IDEAL_WIND_REFERENCE_MPH, color="gray", linestyle=":", linewidth=1)
plt.title("Partial Dependence: Wind Effect vs Ideal")
plt.xlabel("Wind speed set in counterfactual (mph)")
plt.ylabel("Estimated wind effect vs ideal (strokes)")
plt.tight_layout()
plt.show()

In [ ]:
# -----------------------------------------------------------------------------
# Temperature effect vs ideal PDP
# -----------------------------------------------------------------------------

temperature_grid_c = [0, 5, 10, 12, 15, 20, 25, 30, 35]


def update_temperature_actual_df(cf_df: pd.DataFrame, grid_value_c: float) -> pd.DataFrame:
    out = cf_df.copy()
    out["round_temp_c_mean"] = grid_value_c
    return out


def update_temperature_reference_df(cf_df: pd.DataFrame, grid_value_c: float) -> pd.DataFrame:
    out = cf_df.copy()
    out["round_temp_c_mean"] = IDEAL_TEMPERATURE_REFERENCE_C
    return out


temperature_effect_pdp_df = build_effect_pdp_curve(
    base_df=pdp_base_df,
    model=model,
    feature_cols=feature_cols,
    categorical_features=categorical_features,
    update_actual_fn=update_temperature_actual_df,
    update_reference_fn=update_temperature_reference_df,
    grid_values=temperature_grid_c,
)

temperature_effect_pdp_df["temperature_f"] = (temperature_effect_pdp_df["grid_value"] * 9.0 / 5.0) + 32.0
temperature_effect_pdp_df


In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(
    temperature_effect_pdp_df["temperature_f"],
    temperature_effect_pdp_df["effect_mean"],
    marker="o",
    linewidth=2.5,
    color="#ff7f0e",
)
plt.fill_between(
    temperature_effect_pdp_df["temperature_f"],
    temperature_effect_pdp_df["effect_p10"],
    temperature_effect_pdp_df["effect_p90"],
    color="#ff7f0e",
    alpha=0.15,
)
plt.axhline(0.0, color="black", linestyle="--", linewidth=1)
plt.axvline((IDEAL_TEMPERATURE_REFERENCE_C * 9.0 / 5.0) + 32.0, color="gray", linestyle=":", linewidth=1)
plt.title("Partial Dependence: Temperature Effect vs Ideal")
plt.xlabel("Temperature set in counterfactual (F)")
plt.ylabel("Estimated temperature effect vs ideal (strokes)")
plt.tight_layout()
plt.show()


In [ ]:
# -----------------------------------------------------------------------------
# Temperature predicted-score PDP sanity check
# -----------------------------------------------------------------------------

temperature_score_pdp_df = build_partial_dependence_curve(
    base_df=pdp_base_df,
    model=model,
    feature_cols=feature_cols,
    categorical_features=categorical_features,
    vary_col="round_temp_c_mean",
    grid_values=temperature_grid_c,
    update_fn=update_temperature_actual_df,
)

temperature_score_pdp_df["temperature_f"] = (temperature_score_pdp_df["grid_value"] * 9.0 / 5.0) + 32.0
temperature_score_pdp_df


In [ ]:
# -----------------------------------------------------------------------------
# Temperature predicted-score PDP sanity check
# -----------------------------------------------------------------------------

temperature_score_pdp_df = build_partial_dependence_curve(
    base_df=pdp_base_df,
    model=model,
    feature_cols=feature_cols,
    categorical_features=categorical_features,
    vary_col="round_temp_c_mean",
    grid_values=temperature_grid_c,
    update_fn=update_temperature_actual_df,
)

temperature_score_pdp_df["temperature_f"] = (temperature_score_pdp_df["grid_value"] * 9.0 / 5.0) + 32.0
temperature_score_pdp_df


In [ ]:
# -----------------------------------------------------------------------------
# Precipitation effect vs ideal PDP
# -----------------------------------------------------------------------------

precip_grid_mm = [0.0, 0.25, 0.5, 1.0, 2.0, 5.0]


def update_precip_actual_df(cf_df: pd.DataFrame, grid_value_mm: float) -> pd.DataFrame:
    out = cf_df.copy()
    out["round_precip_mm_sum"] = grid_value_mm
    out["round_precip_mm_mean"] = grid_value_mm
    return out


def update_precip_reference_df(cf_df: pd.DataFrame, grid_value_mm: float) -> pd.DataFrame:
    out = cf_df.copy()
    out["round_precip_mm_sum"] = IDEAL_PRECIP_REFERENCE_MM
    out["round_precip_mm_mean"] = IDEAL_PRECIP_REFERENCE_MM
    return out


precip_effect_pdp_df = build_effect_pdp_curve(
    base_df=pdp_base_df,
    model=model,
    feature_cols=feature_cols,
    categorical_features=categorical_features,
    update_actual_fn=update_precip_actual_df,
    update_reference_fn=update_precip_reference_df,
    grid_values=precip_grid_mm,
)

precip_effect_pdp_df["precip_mm"] = precip_effect_pdp_df["grid_value"]
precip_effect_pdp_df


In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(
    precip_effect_pdp_df["precip_mm"],
    precip_effect_pdp_df["effect_mean"],
    marker="o",
    linewidth=2.5,
    color="#2ca02c",
)
plt.fill_between(
    precip_effect_pdp_df["precip_mm"],
    precip_effect_pdp_df["effect_p10"],
    precip_effect_pdp_df["effect_p90"],
    color="#2ca02c",
    alpha=0.15,
)
plt.axhline(0.0, color="black", linestyle="--", linewidth=1)
plt.axvline(IDEAL_PRECIP_REFERENCE_MM, color="gray", linestyle=":", linewidth=1)
plt.title("Partial Dependence: Precipitation Effect vs Ideal")
plt.xlabel("Precipitation set in counterfactual (mm)")
plt.ylabel("Estimated precipitation effect vs ideal (strokes)")
plt.tight_layout()
plt.show()


In [ ]:
conclusion_summary_df = pd.DataFrame(
    [
        {
            "model_name": "round_one_stage_catboost",
            "model_variant": experiment["name"],
            "rows_scored": int(len(scored_df)),
            "train_rmse": float(model_metrics.loc[model_metrics["split"] == "train", "rmse"].iloc[0]),
            "valid_rmse": float(model_metrics.loc[model_metrics["split"] == "valid", "rmse"].iloc[0]),
            "test_rmse": float(model_metrics.loc[model_metrics["split"] == "test", "rmse"].iloc[0]),
            "ideal_wind_reference_mph": IDEAL_WIND_REFERENCE_MPH,
            "ideal_wind_gust_reference_mph": IDEAL_WIND_GUST_REFERENCE_MPH,
            "ideal_temperature_reference_c": IDEAL_TEMPERATURE_REFERENCE_C,
            "ideal_precip_reference_mm": IDEAL_PRECIP_REFERENCE_MM,
            "avg_wind_effect_vs_ideal": float(scored_df["estimated_wind_effect_vs_ideal"].mean()),
            "avg_temperature_effect_vs_ideal": float(scored_df["estimated_temperature_effect_vs_ideal"].mean()),
            "avg_precip_effect_vs_ideal": float(scored_df["estimated_precip_effect_vs_ideal"].mean()),
            "avg_total_weather_effect_vs_ideal": float(scored_df["estimated_total_weather_effect_vs_ideal"].mean()),
        }
    ]
)

conclusion_summary_df


In [ ]:
# -----------------------------------------------------------------------------
# Wind effect vs ideal PDP
# -----------------------------------------------------------------------------

wind_grid_mph = [0, 2, 5, 8, 12, 16, 20, 25, 30]
wind_grid_mps = [x * MPH_TO_MPS for x in wind_grid_mph]

IDEAL_GUST_GAP_MPH = max(0.0, IDEAL_WIND_GUST_REFERENCE_MPH - IDEAL_WIND_REFERENCE_MPH)


def candidate_gust_from_wind_mph(wind_speed_mph: float) -> float:
    return max(wind_speed_mph, wind_speed_mph + IDEAL_GUST_GAP_MPH)


def update_wind_actual_df(cf_df: pd.DataFrame, grid_value_mps: float) -> pd.DataFrame:
    out = cf_df.copy()

    wind_speed_mph = grid_value_mps * MPS_TO_MPH
    wind_gust_mph = candidate_gust_from_wind_mph(wind_speed_mph)

    out["round_wind_speed_mps_mean"] = grid_value_mps
    out["round_wind_speed_mps_max"] = grid_value_mps
    out["round_wind_gust_mps_mean"] = wind_gust_mph * MPH_TO_MPS
    out["round_wind_gust_mps_max"] = wind_gust_mph * MPH_TO_MPS

    out["round_wind_speed_bucket"] = wind_speed_bucket_from_mph(wind_speed_mph)
    out["round_wind_gust_bucket"] = wind_gust_bucket_from_mph(wind_gust_mph)
    return out


def update_wind_reference_df(cf_df: pd.DataFrame, grid_value_mps: float) -> pd.DataFrame:
    out = cf_df.copy()

    out["round_wind_speed_mps_mean"] = IDEAL_WIND_REFERENCE_MPH * MPH_TO_MPS
    out["round_wind_speed_mps_max"] = IDEAL_WIND_REFERENCE_MPH * MPH_TO_MPS
    out["round_wind_gust_mps_mean"] = IDEAL_WIND_GUST_REFERENCE_MPH * MPH_TO_MPS
    out["round_wind_gust_mps_max"] = IDEAL_WIND_GUST_REFERENCE_MPH * MPH_TO_MPS

    out["round_wind_speed_bucket"] = wind_speed_bucket_from_mph(IDEAL_WIND_REFERENCE_MPH)
    out["round_wind_gust_bucket"] = wind_gust_bucket_from_mph(IDEAL_WIND_GUST_REFERENCE_MPH)
    return out


wind_effect_pdp_df = build_effect_pdp_curve(
    base_df=pdp_base_df,
    model=model,
    feature_cols=feature_cols,
    categorical_features=categorical_features,
    update_actual_fn=update_wind_actual_df,
    update_reference_fn=update_wind_reference_df,
    grid_values=wind_grid_mps,
)

wind_effect_pdp_df["wind_speed_mph"] = wind_effect_pdp_df["grid_value"] * MPS_TO_MPH
wind_effect_pdp_df
